# 🔤 Урок 17 — Самостоятельная работа: как устроен ChatGPT

### Как работать
Читай 📖 · запускай ▶️ · выполняй задания 📝 · отвечай словами ✍️.

> 🎯 К концу урока ты сам покажешь, как **твоё** предложение превращается в числа, и объяснишь, что такое внимание.

Задания трёх уровней: 🟢 базовый · 🔵 продвинутый · ⭐ со звёздочкой.
Каждая строка кода подписана — читай комментарии справа от `#`.

## Шаг 1 · Текст → числа (настоящие токены)
Модель режет текст не на слова, а на кусочки-**токены**. Запусти и увидишь свои токены.

In [ ]:
# Устанавливаем настоящий токенайзер GPT (нужен интернет — в Colab он есть)
!pip install tiktoken -q
import tiktoken                                  # библиотека токенизации
enc = tiktoken.get_encoding("cl100k_base")       # тот самый токенайзер, что у GPT-4

def показать(текст):                             # функция: печатает токены для текста
    номера = enc.encode(текст)                   # текст -> список номеров-токенов
    куски  = [enc.decode([n]) for n in номера]   # каждый номер -> обратно в кусочек текста
    print(f'{текст!r}  ->  {len(номера)} токенов')  # сколько всего токенов
    print('   кусочки:', куски)                  # из каких кусочков собран текст

показать('машинное обучение')                    # длинные слова рвутся на части
показать('machine learning')                     # по-английски тех же слов меньше

### 📝 Задание 🟢 базовый
Впиши **своё имя** и **своё любимое предложение** и запусти.

In [ ]:
показать('__ВПИШИ_СВОЁ_ИМЯ__')               # напр. показать('Тимур')
показать('__ВПИШИ_СВОЁ_ПРЕДЛОЖЕНИЕ__')       # напр. показать('Я учусь машинному обучению')

✍️ **Ответь.** Возьми одну мысль по-русски и по-английски (напр. «нейросеть» / «neural network»).
Где токенов больше? Как думаешь, почему за русский запрос к платному API платят больше?

*Ответ:* …

## Шаг 2 · Эмбеддинги — близкие по смыслу слова = близкие числа
Каждое слово — это вектор (список чисел). Похожие по смыслу слова стоят рядом.
Числа-координаты тут придуманы для примера, **но близость мы считаем по-настоящему.**

In [ ]:
import numpy as np                            # библиотека для чисел и векторов

# Словарь: слово -> его вектор-эмбеддинг (4 числа для наглядности)
emb = {
    'король':  np.array([0.90, 0.80, 0.10, 0.70]),
    'королева':np.array([0.90, 0.20, 0.10, 0.75]),
    'мужчина': np.array([0.85, 0.90, 0.15, 0.10]),
    'женщина': np.array([0.85, 0.15, 0.15, 0.12]),
    'банан':   np.array([0.10, 0.40, 0.90, 0.05]),
}

def близость(a, b):                             # косинусная близость: 1=похожи, 0=нет
    va, vb = emb[a], emb[b]                      # берём два вектора
    return va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb))  # формула косинуса

print('король–королева:', round(близость('король','королева'), 2))  # должно быть высоко
print('король–банан   :', round(близость('король','банан'),    2))  # должно быть низко

### 📝 Задание 🔵 продвинутый
Добавь в `emb` **свои 2 слова** (придумай им координаты) и найди самую близкую пару.

In [ ]:
emb['яблоко'] = np.array([0.20, 0.30, 0.86, 0.10])   # добавили слово-вектор
emb['груша']  = np.array([0.22, 0.31, 0.85, 0.11])   # ещё одно похожее на фрукт

# Проверь несколько пар сам:
print('банан–яблоко :', round(близость('банан','яблоко'), 2))
print('яблоко–груша :', round(близость('яблоко','груша'), 2))
# ✍️ Какая пара оказалась ближе всего и почему? Ответ: ...

### 📝 Задание ⭐ со звёздочкой — арифметика смыслов
Модель ловит **отношения** между словами. Запусти — ответ вычислится, а не будет вписан заранее.

In [ ]:
цель = emb['король'] - emb['мужчина'] + emb['женщина']   # убрали «мужское», добавили «женское»

лучший, счёт = None, -1                         # ищем ближайшее слово к «цели»
for слово, вектор in emb.items():               # перебираем все слова
    if слово in ('король','мужчина','женщина'): continue   # эти три пропускаем
    c = цель @ вектор / (np.linalg.norm(цель) * np.linalg.norm(вектор))  # близость
    if c > счёт: лучший, счёт = слово, c         # запоминаем самое близкое

print('король − мужчина + женщина  ≈ ', лучший)  # получится «королева»

## Шаг 3 · Внимание — на какие слова смотрит модель
Чтобы понять слово «она», модель смотрит на все слова и решает, какое сейчас важнее.
Строка = слово; чем темнее клетка, тем сильнее оно «смотрит» на слово-столбец.

In [ ]:
import numpy as np, matplotlib.pyplot as plt   # числа + рисование

слова = ['Кошка','гналась','за','мышью','потому','что','она','голодной']
# Иллюстративная карта внимания (настоящую даёт модель). Каждая строка в сумме ~1.
A = np.array([
 [.50,.10,.05,.15,.02,.03,.10,.05],   # Кошка
 [.20,.30,.10,.25,.03,.02,.05,.05],   # гналась
 [.05,.15,.40,.25,.05,.05,.03,.02],   # за
 [.15,.20,.10,.40,.05,.03,.05,.02],   # мышью
 [.05,.05,.05,.05,.40,.25,.10,.05],   # потому
 [.05,.05,.05,.05,.30,.35,.10,.05],   # что
 [.55,.05,.03,.07,.03,.03,.14,.10],   # она  -> сильнее всего на «Кошку»
 [.40,.05,.03,.12,.05,.05,.25,.05],   # голодной
])
plt.figure(figsize=(6,5))                        # размер картинки
plt.imshow(A, cmap='Purples', vmin=0, vmax=1)    # рисуем карту (фиолетовый = сильнее)
plt.xticks(range(8), слова, rotation=45, ha='right')  # подписи по столбцам
plt.yticks(range(8), слова)                      # подписи по строкам
plt.title('Строка = слово смотрит на слова-столбцы')
plt.colorbar(); plt.tight_layout(); plt.show()   # шкала цвета и показ

✍️ **Вопрос.** Посмотри на строку «она» — на какое слово она смотрит сильнее всего?
А если бы в конце было «быстрой» вместо «голодной» — на кого смотрела бы «она»? Почему?

*Ответ:* …

## 🧠 Галлюцинации
Модель подбирает **правдоподобное** продолжение, а не проверяет правду.
Поэтому может уверенно выдумать факт — как сеть из урока 14 уверенно называла цифру на шуме.
**Правило:** важные цифры, имена, даты — всегда перепроверяй.

## ✅ Проверь себя
1. Что такое токен и почему он не всегда равен слову?
2. Что значит «близкие по смыслу слова — близкие числа»?
3. Что показывает внимание своими словами?
4. Почему модель может уверенно выдать ложный факт?

<details><summary>Ответы</summary>

1. Кусочек текста (слово или его часть) со своим номером; длинные слова рвутся на несколько токенов.
2. У похожих по смыслу слов похожие векторы-эмбеддинги, поэтому они рядом на «карте смыслов».
3. На какие слова модель смотрит сильнее, когда обрабатывает текущее слово.
4. Она подбирает правдоподобное продолжение, а не сверяется с базой фактов.
</details>

## 🏁 Задание ⭐ для сильных — BertViz на настоящей модели
Запусти и наведи на **`it`** — увидишь связь со словом **`cat`**.
Потом смени `hungry` на `fast` и посмотри, как внимание к `it` перескакивает на `mouse`.

In [ ]:
# Только в Colab с интернетом
!pip install bertviz transformers -q
from bertviz import head_view
from transformers import AutoTokenizer, AutoModel
tok = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
inputs = tok('The cat chased the mouse because it was hungry', return_tensors='pt')
head_view(model(**inputs).attentions, tok.convert_ids_to_tokens(inputs['input_ids'][0]))

---
### 🎉 Готово!
Ты сам показал, как **твоё** предложение становится числами, и на что «смотрит» модель.
Дальше (урок 18) — заставим модель работать: классифицировать отзывы и сделаем бота с характером!